# AI-Powered Flood Risk Prediction
# Logistic Regression
## Manuela Munoz Ramirez

Trains a Logistic Regression model to classify flood risk at Murray Bridge using real
historical river level data from the Murray–Darling Basin Authority (MDBA).

In [5]:
!pip install pandas numpy scikit-learn joblib -q


## 1. Load real data


In [6]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, matthews_corrcoef, classification_report
import joblib

url = "https://raw.githubusercontent.com/ale-navarro23/flood-risk-prototype/main/data/murray_bridge_river_level_historical.csv"
df = pd.read_csv(url, skiprows=4, names=["datetime", "water_level_m", "conductivity", "water_temp_c"])
df["datetime"] = pd.to_datetime(df["datetime"], format="%H:%M:%S %d/%m/%Y", errors="coerce")
df = df.dropna(subset=["datetime", "water_level_m"]).sort_values("datetime").reset_index(drop=True)

print("Rows:", len(df))
print("Date range:", df["datetime"].min(), "to", df["datetime"].max())
df.head()

Rows: 6364
Date range: 2009-01-08 09:00:00 to 2026-07-02 09:00:00


,datetime,water_level_m,conductivity,water_temp_c
0,2009-01-08 09:00:00,-0.584,680.492,22.788
1,2009-01-09 09:00:00,-0.640,682.482,22.719
2,2009-01-10 09:00:00,-0.662,685.370,22.657
3,2009-01-11 09:00:00,-0.637,691.286,22.730
4,2009-01-12 09:00:00,-0.649,691.175,22.737


## 2. Feature engineering


In [7]:
df["level_lag1"] = df["water_level_m"].shift(1)
df["level_lag2"] = df["water_level_m"].shift(2)
df["level_roll7"] = df["water_level_m"].shift(1).rolling(7).mean()
df["level_change3"] = df["water_level_m"].shift(1) - df["water_level_m"].shift(4)
df = df.dropna(subset=["level_lag1", "level_lag2", "level_roll7", "level_change3"])

threshold = df["water_level_m"].quantile(0.80)
df["high_risk"] = (df["water_level_m"] >= threshold).astype(int)

print("Risk threshold (m):", round(threshold, 3))
df["high_risk"].value_counts()

Risk threshold (m): 0.806


,count
high_risk,
0,5083
1,1274


## 3. Train/test split


In [8]:
features = ["level_lag1", "level_lag2", "level_roll7", "level_change3"]
X = df[features]
y = df["high_risk"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

## 4. Train Logistic Regression


In [9]:
model_lr = LogisticRegression(class_weight="balanced", max_iter=1000)
model_lr.fit(X_train, y_train)
pred_lr = model_lr.predict(X_test)
f1_lr = f1_score(y_test, pred_lr)
mcc_lr = matthews_corrcoef(y_test, pred_lr)
print(f"F1: {f1_lr:.3f}  MCC: {mcc_lr:.3f}")
print(classification_report(y_test, pred_lr))

F1: 0.778  MCC: 0.726
              precision    recall  f1-score   support

           0       0.98      0.89      0.93      1017
           1       0.67      0.92      0.78       255

    accuracy                           0.89      1272
   macro avg       0.83      0.90      0.85      1272
weighted avg       0.92      0.89      0.90      1272



## 5. Save trained model

In [10]:
joblib.dump(model_lr, "logistic_regression_real.joblib")
print("Model saved.")

Model saved.
